In [1]:
# ==========================================
# CELLULE 1 : Importations et Chargement
# ==========================================
import pandas as pd
import numpy as np

# Le chemin vers ton fichier génomique (Vérifie qu'il est correct dans ton panneau de droite)
chemin_fichier = '/kaggle/input/datasets/samdemharter/brca-multiomics-tcga/brca_data_w_subtypes.csv'

# Lecture du fichier
df_genes = pd.read_csv(chemin_fichier)

print("✅ Fichier génomique chargé avec succès !")

✅ Fichier génomique chargé avec succès !


In [2]:
# ==========================================
# CELLULE 2 : Exploration Rapide
# ==========================================
# 1. On regarde la taille du tableau (Nombre de patients, Nombre de gènes)
print(f"Dimensions du dataset : {df_genes.shape}")

# 2. On affiche les 5 premières lignes et les 10 premières colonnes pour ne pas saturer l'écran
print("\nAperçu des données :")
display(df_genes.iloc[:5, :10])

Dimensions du dataset : (705, 1941)

Aperçu des données :


,rs_CLEC3A,rs_CPB1,rs_SCGB2A2,rs_SCGB1D2,rs_TFF1,rs_MUCL1,rs_GSTM1,rs_PIP,rs_ADIPOQ,rs_ADH1B
0,0.892818,6.580103,14.123672,10.606501,13.189237,6.649466,10.520335,10.338490,10.248379,10.229970
1,0.000000,3.691311,17.116090,15.517231,9.867616,9.691667,8.179522,7.911723,1.289598,1.818891
2,3.748150,4.375255,9.658123,5.326983,12.109539,11.644307,10.517330,5.114925,11.975349,11.911437
3,0.000000,18.235519,18.535480,14.533584,14.078992,8.913760,10.557465,13.304434,8.205059,9.211476
4,0.000000,4.583724,15.711865,12.804521,8.881669,8.430028,12.964607,6.806517,4.294341,5.385714


In [3]:
# ==========================================
# CELLULE 3 : Qualité des données et Étiquettes
# ==========================================

# 1. Chasse aux valeurs manquantes
total_nan = df_genes.isnull().sum().sum()
print(f"⚠️ Nombre total de valeurs manquantes (NaN) dans tout le tableau : {total_nan}")

# 2. Identification des métadonnées (ce qui n'est pas un nombre)
# Les gènes sont généralement des chiffres (float64). Ce qui est en texte (object) nous intéresse !
colonnes_texte = df_genes.select_dtypes(include=['object']).columns.tolist()

print(f"\n🔍 Colonnes contenant du texte (Identifiants / Étiquettes) :")
for col in colonnes_texte:
    print(f" - {col}")

⚠️ Nombre total de valeurs manquantes (NaN) dans tout le tableau : 389

🔍 Colonnes contenant du texte (Identifiants / Étiquettes) :
 - PR.Status
 - ER.Status
 - HER2.Final.Status
 - histological.type


In [5]:
# ==========================================
# CELLULE 3 : Préparation et Normalisation
# ==========================================
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# 1. On sépare les métadonnées (colonnes de texte) et la matrice de gènes (numérique)
colonnes_cliniques = ['PR.Status', 'ER.Status', 'HER2.Final.Status', 'histological.type']

# On isole toutes les colonnes numériques qui représentent les gènes
df_genes_numerique = df_genes.select_dtypes(include=[np.number]).copy()

print(f"🧬 Nombre de gènes / caractéristiques numériques à traiter : {df_genes_numerique.shape[1]}")

# 2. Gestion des valeurs manquantes (Imputation par la médiane)
imputeur = SimpleImputer(strategy='median')
matrice_imputee = imputeur.fit_transform(df_genes_numerique)

# 3. Normalisation (Centrage et réduction : moyenne=0, écart-type=1)
scaler = StandardScaler()
matrice_normalisee = scaler.fit_transform(matrice_imputee)

# Conversion en tenseur PyTorch pour la suite
X_genomique = torch.tensor(matrice_normalisee, dtype=torch.float32)

print(f"✅ Nettoyage terminé ! Tenseur prêt pour l'Auto-encodeur. Dimensions : {X_genomique.shape}")
print(f"⚠️ Vérification finale des NaN : {torch.isnan(X_genomique).sum().item()} valeur manquante restante.")

🧬 Nombre de gènes / caractéristiques numériques à traiter : 1937
✅ Nettoyage terminé ! Tenseur prêt pour l'Auto-encodeur. Dimensions : torch.Size([705, 1937])
⚠️ Vérification finale des NaN : 0 valeur manquante restante.


In [6]:
# ==========================================
# CELLULE 4 : Architecture de l'Auto-Encodeur
# ==========================================
import torch.nn as nn

class AutoEncodeurGenomique(nn.Module):
    def __init__(self, n_genes):
        super(AutoEncodeurGenomique, self).__init__()
        
        # 1. L'ENCODEUR (Compression)
        # On passe de 1941 -> 512 -> 128
        self.encodeur = nn.Sequential(
            nn.Linear(n_genes, 512),
            nn.ReLU(),           # Fonction d'activation (pour comprendre la biologie non-linéaire)
            nn.Dropout(0.2),     # On désactive 20% des neurones au hasard pour éviter le surapprentissage
            nn.Linear(512, 128), # LE GOULOT D'ÉTRANGLEMENT (Espace latent)
            nn.ReLU()
        )
        
        # 2. LE DÉCODEUR (Reconstruction)
        # On passe de 128 -> 512 -> 1941
        self.decodeur = nn.Sequential(
            nn.Linear(128, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, n_genes) # Retour à la dimension d'origine (1941)
        )

    def forward(self, x):
        # Le chemin de la donnée à travers le modèle
        signature_compressee = self.encodeur(x)
        reconstruction = self.decodeur(signature_compressee)
        return reconstruction, signature_compressee

# 3. Initialisation du modèle sur la carte graphique
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"💻 Processeur utilisé : {device}")

nombre_genes_input = X_genomique.shape[1] # 1941
modele_genes = AutoEncodeurGenomique(n_genes=nombre_genes_input).to(device)

# 4. Le Moteur et la Boussole
# On utilise la MSE (Mean Squared Error) car on essaie de reconstruire des chiffres exacts
critere_reconstruction = nn.MSELoss()
optimiseur_genes = torch.optim.Adam(modele_genes.parameters(), lr=0.001)

print(f"✅ Architecture de l'Auto-encodeur instanciée !")
print(f"L'entonnoir va compresser {nombre_genes_input} gènes en une signature de 128 valeurs.")

💻 Processeur utilisé : cuda
✅ Architecture de l'Auto-encodeur instanciée !
L'entonnoir va compresser 1937 gènes en une signature de 128 valeurs.


In [7]:
# ==========================================
# CELLULE 5 : Entraînement de l'Auto-Encodeur
# ==========================================
from torch.utils.data import TensorDataset, DataLoader

# 1. Préparation du convoyeur de données
# L'astuce absolue de l'Auto-Encodeur : l'entrée (x) et la cible (y) sont exactement les mêmes !
dataset_genes = TensorDataset(X_genomique, X_genomique)
loader_genes = DataLoader(dataset_genes, batch_size=32, shuffle=True)

epochs = 50

print("🚀 Début de l'entraînement de compression génomique...")

for epoch in range(epochs):
    modele_genes.train()
    train_loss = 0.0
    
    for batch_x, batch_y in loader_genes:
        # On envoie les gènes originaux sur la carte graphique
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device) # Rappel : c'est la même chose que batch_x !
        
        optimiseur_genes.zero_grad()
        
        # Le modèle compresse (signature) puis tente de décompresser (reconstruction)
        reconstruction, signature = modele_genes(batch_x)
        
        # La boussole calcule la différence entre la copie et l'original
        perte = critere_reconstruction(reconstruction, batch_y)
        
        perte.backward()
        optimiseur_genes.step()
        
        train_loss += perte.item()
        
    avg_loss = train_loss / len(loader_genes)
    
    # Affichage de la progression toutes les 10 epochs pour garder une console propre
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Étape {epoch+1:02d}/{epochs} | Erreur de reconstruction (Loss) : {avg_loss:.4f}")

print("\n🏁 Entraînement terminé ! L'Auto-Encodeur sait maintenant compresser les profils tumoraux.")

🚀 Début de l'entraînement de compression génomique...
Étape 01/50 | Erreur de reconstruction (Loss) : 0.9149
Étape 10/50 | Erreur de reconstruction (Loss) : 0.5365
Étape 20/50 | Erreur de reconstruction (Loss) : 0.4395
Étape 30/50 | Erreur de reconstruction (Loss) : 0.3908
Étape 40/50 | Erreur de reconstruction (Loss) : 0.3819
Étape 50/50 | Erreur de reconstruction (Loss) : 0.3629

🏁 Entraînement terminé ! L'Auto-Encodeur sait maintenant compresser les profils tumoraux.


In [8]:
# ==========================================
# CELLULE 6 : Extraction de l'Espace Latent (Signatures)
# ==========================================

# 1. On met le modèle en mode "Examen" (il n'apprend plus)
modele_genes.eval()

# 2. On désactive le calcul des gradients pour économiser la mémoire (très important !)
with torch.no_grad():
    # On fait passer TOUS les patients uniquement dans l'ENCODEUR
    # On obtient notre matrice compressée de 128 colonnes
    signatures_compressees = modele_genes.encodeur(X_genomique.to(device))
    
    # On ramène les résultats sur le CPU et on les convertit en tableau Numpy
    signatures_numpy = signatures_compressees.cpu().numpy()

# 3. On transforme ça en un beau tableau Pandas (DataFrame)
# On crée des noms de colonnes automatiques : "Gene_Feature_1", "Gene_Feature_2", etc.
noms_colonnes = [f"Gene_Feature_{i+1}" for i in range(signatures_numpy.shape[1])]
df_signatures = pd.DataFrame(signatures_numpy, columns=noms_colonnes)

# On récupère l'index (les identifiants des patients) du tableau d'origine pour ne pas les perdre
df_signatures.index = df_genes.index

print(f"✅ Extraction réussie ! L'ancien tableau (705, 1941) est devenu : {df_signatures.shape}")
print("\nAperçu des 5 premières super-signatures génomiques :")
display(df_signatures.head())

✅ Extraction réussie ! L'ancien tableau (705, 1941) est devenu : (705, 128)

Aperçu des 5 premières super-signatures génomiques :


,Gene_Feature_1,Gene_Feature_2,Gene_Feature_3,Gene_Feature_4,Gene_Feature_5,Gene_Feature_6,Gene_Feature_7,Gene_Feature_8,Gene_Feature_9,Gene_Feature_10,...,Gene_Feature_119,Gene_Feature_120,Gene_Feature_121,Gene_Feature_122,Gene_Feature_123,Gene_Feature_124,Gene_Feature_125,Gene_Feature_126,Gene_Feature_127,Gene_Feature_128
0,0.0,1.857862,0.000000,0.000000,0.0,0.0,2.275110,0.000000,0.0,0.0,...,2.661347,0.0,0.0,0.0,4.876503,1.193628,3.667185,5.432762,0.0,6.541193
1,0.0,2.216990,0.000000,0.735747,0.0,0.0,0.000000,3.197512,0.0,0.0,...,0.000000,0.0,0.0,0.0,2.440278,5.531266,1.514261,1.010921,0.0,1.254429
2,0.0,0.000000,0.000000,0.000000,0.0,0.0,10.644073,0.606668,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.000000,0.000000,2.037649,10.375899,0.0,7.020140
3,0.0,5.024593,0.000000,0.000000,0.0,0.0,1.827483,12.165783,0.0,0.0,...,1.543537,0.0,0.0,0.0,1.400414,0.000000,3.842348,0.000000,0.0,9.613652
4,0.0,0.000000,0.389999,2.242688,0.0,0.0,4.104432,0.000000,0.0,0.0,...,0.000000,0.0,0.0,0.0,3.178823,0.911332,8.825245,6.054854,0.0,5.197796


In [13]:
# ==========================================
# CELLULE 7 : Récupération des Étiquettes de Survie
# ==========================================

# 1. On extrait la colonne de survie du fichier d'origine de Sam Demharter
etiquettes_survie = df_genes['vital.status'].copy()

# 2. On l'ajoute directement à nos signatures (les 128 colonnes générées par l'Auto-encodeur)
df_final = df_signatures.copy()

# On la nomme avec un tiret du bas pour être consistant avec le Pilier 1
df_final['vital_status'] = etiquettes_survie

print("✅ Récupération réussie ! Pas besoin de fusionner les tableaux.")
print(f"Dimensions de notre tableau final prêt pour le réseau de neurones : {df_final.shape}")
print("\nRépartition de la survie (0 = Vivant, 1 = Mort) :")
print(df_final['vital_status'].value_counts())

✅ Récupération réussie ! Pas besoin de fusionner les tableaux.
Dimensions de notre tableau final prêt pour le réseau de neurones : (705, 129)

Répartition de la survie (0 = Vivant, 1 = Mort) :
vital_status
0    611
1     94
Name: count, dtype: int64


In [14]:
# ==========================================
# CELLULE 8 : Découpage Entraînement / Validation
# ==========================================
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
import torch

# 1. On sépare les caractéristiques (les 128 gènes compressés) et la cible (la survie)
X = df_final.drop(columns=['vital_status']).values
y = df_final['vital_status'].values

# 2. Découpage 80% / 20%
# L'argument 'stratify=y' est VITAL : il s'assure qu'on garde le même pourcentage 
# de patients décédés dans le groupe d'entraînement et le groupe de validation.
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Conversion au format tenseur pour la carte graphique (PyTorch)
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1) # Format [batch_size, 1]

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

# 4. Création des convoyeurs (DataLoaders)
train_dataset_mlp = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset_mlp = TensorDataset(X_val_tensor, y_val_tensor)

# batch_size=32 : on traite 32 patients à la fois
train_loader_mlp = DataLoader(train_dataset_mlp, batch_size=32, shuffle=True)
val_loader_mlp = DataLoader(val_dataset_mlp, batch_size=32, shuffle=False)

print("✅ Données découpées et prêtes pour le réseau de classification !")
print(f"📊 Patients pour l'Entraînement : {len(X_train)} | Patients pour l'Examen : {len(X_val)}")

✅ Données découpées et prêtes pour le réseau de classification !
📊 Patients pour l'Entraînement : 564 | Patients pour l'Examen : 141


In [15]:
# ==========================================
# CELLULE 9 : L'Expert Génomique (MLP) et Entraînement
# ==========================================
import torch.nn as nn
import torch.optim as optim
import time

# 1. L'Architecture du Réseau
class ClassifieurSurvie(nn.Module):
    def __init__(self, input_dim=128):
        super(ClassifieurSurvie, self).__init__()
        self.reseau = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),      # On coupe 30% des neurones pour forcer le modèle à généraliser
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)      # 1 seul neurone à la fin : le score de risque
        )

    def forward(self, x):
        return self.reseau(x)

# On instancie le modèle sur le GPU
modele_mlp = ClassifieurSurvie(input_dim=128).to(device)

# La boussole (BCE pour du binaire 0/1) et le moteur (Adam)
critere_mlp = nn.BCEWithLogitsLoss()
optimiseur_mlp = optim.Adam(modele_mlp.parameters(), lr=0.001)

# 2. La Boucle d'Entraînement
epochs_mlp = 30
print("🚀 Lancement de l'entraînement de l'Expert Génomique...")
start_time = time.time()

for epoch in range(epochs_mlp):
    
    # --- PHASE D'ENTRAÎNEMENT ---
    modele_mlp.train()
    train_loss = 0.0
    
    for batch_x, batch_y in train_loader_mlp:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimiseur_mlp.zero_grad()
        predictions = modele_mlp(batch_x)
        perte = critere_mlp(predictions, batch_y)
        
        perte.backward()
        optimiseur_mlp.step()
        train_loss += perte.item()
        
    avg_train_loss = train_loss / len(train_loader_mlp)

    # --- PHASE D'EXAMEN (Validation) ---
    modele_mlp.eval()
    val_loss = 0.0
    predictions_correctes = 0
    total_val = 0
    
    with torch.no_grad():
        for batch_x, batch_y in val_loader_mlp:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            predictions = modele_mlp(batch_x)
            perte = critere_mlp(predictions, batch_y)
            val_loss += perte.item()
            
            # Calcul de la précision clinique (Probabilité > 50%)
            probabilites = torch.sigmoid(predictions)
            predictions_finales = (probabilites > 0.5).float()
            
            predictions_correctes += (predictions_finales == batch_y).sum().item()
            total_val += batch_y.size(0)
            
    avg_val_loss = val_loss / len(val_loader_mlp)
    accuracy = (predictions_correctes / total_val) * 100
    
    # On n'affiche que toutes les 5 epochs pour ne pas polluer l'écran
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Étape {epoch+1:02d}/{epochs_mlp} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Précision: {accuracy:.2f}%")

temps_ecoule = time.time() - start_time
print(f"\n🏁 Entraînement terminé en {temps_ecoule:.2f} secondes !")

🚀 Lancement de l'entraînement de l'Expert Génomique...
Étape 01/30 | Train Loss: 0.5108 | Val Loss: 0.4481 | Précision: 86.52%
Étape 05/30 | Train Loss: 0.3509 | Val Loss: 0.4306 | Précision: 86.52%
Étape 10/30 | Train Loss: 0.2789 | Val Loss: 0.4362 | Précision: 87.94%
Étape 15/30 | Train Loss: 0.1967 | Val Loss: 0.4768 | Précision: 88.65%
Étape 20/30 | Train Loss: 0.1381 | Val Loss: 0.5088 | Précision: 90.07%
Étape 25/30 | Train Loss: 0.0860 | Val Loss: 0.6714 | Précision: 87.94%
Étape 30/30 | Train Loss: 0.0573 | Val Loss: 0.7847 | Précision: 87.94%

🏁 Entraînement terminé en 1.41 secondes !


In [ ]:
# ==========================================
# CELLULE 10 : Sauvegarde des Modèles Génomiques
# ==========================================

# Sauvegarde de l'entonnoir (pour compresser de nouveaux patients le jour J)
torch.save(modele_genes.state_dict(), "autoencodeur_genomique.pth")

# Sauvegarde du classifieur (pour prédire la survie à partir des gènes compressés)
torch.save(modele_mlp.state_dict(), "mlp_survie_genomique.pth")

print("💾 SUCCÈS ! Les deux cerveaux génomiques sont sauvegardés :")
print(" 1. autoencodeur_genomique.pth")
print(" 2. mlp_survie_genomique.pth")